# Message Content Blocks

The `content.py` module defines standardized, provider-independent content blocks for Large Language Model input and output. 

A message may contain an ordered list of text, reasoning, tool-call, citation, image, audio, video, plaintext, file, and provider-specific blocks.

Provider-specific metadata can be stored in the `extras` field of standard blocks. Data that does not match a standard block can be stored in a `NonStandardContentBlock`.

## Constants

1. `KNOWN_BLOCK_TYPES`: Stores block-type names recognized by `langchain-core >= 1.0.0`.

   A block whose type is not in this set is treated as provider-specific.

   * **Definition:**
     ```python
     KNOWN_BLOCK_TYPES = {
         "text",
         "reasoning",
         "tool_call",
         "invalid_tool_call",
         "tool_call_chunk",
         "image",
         "audio",
         "file",
         "text-plain",
         "video",
         "server_tool_call",
         "server_tool_call_chunk",
         "server_tool_result",
         "non_standard",
     }
     ```

In [3]:
from langchain_core.messages.content import KNOWN_BLOCK_TYPES
block = {"type": "custom_chart"}
print(
    "Standard block"
    if block["type"] in KNOWN_BLOCK_TYPES
    else "Provider-specific block"
)

Provider-specific block


# Citation: `TypedDict`
`Citation` represents an annotation that links part of a model response to a source document. Its start and end indices refer to the response text, not the original source text.
### Fields
1. `type`:`Literal["citation"]`:= Identifies the annotation as a citation.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier for the citation.
3. `url`:`NotRequired[str]`:= Stores the URL of the cited document.
4. `title`:`NotRequired[str]`:= Stores the title of the cited document.
5. `start_index`:`NotRequired[int]`:= Stores the starting index in the response text where the citation applies.
6. `end_index`:`NotRequired[int]`:= Stores the ending index in the response text where the citation applies.
7. `cited_text`:`NotRequired[str]`:= Stores an excerpt from the cited source.
8. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.

In [4]:
from langchain_core.messages import Citation

response = "Paris is the capital of France."

start = response.index("Paris")
end = start + len("Paris")

citation = Citation(
    type="citation",
    title="Paris",
    url="https://en.wikipedia.org/wiki/Paris",
    start_index=start,
    end_index=end,
    cited_text="Paris",
)

print("Response:", response)
print("Cited part:", response[citation["start_index"]:citation["end_index"]])
print("Source:", citation["url"])

# Here, start_index and end_index locate the cited portion inside the model response text.

Response: Paris is the capital of France.
Cited part: Paris
Source: https://en.wikipedia.org/wiki/Paris


# NonStandardAnnotation: `TypedDict`
`NonStandardAnnotation` stores a provider-specific annotation that does not match a standardized annotation type.
## Fields
1. `type`:`Literal["non_standard_annotation"]`:= Identifies the annotation as non-standard.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier.
3. `value`:`dict[str, Any]`:= Stores the provider-specific annotation data.

In [5]:
from langchain_core.messages import NonStandardAnnotation

annotation = NonStandardAnnotation(
    type="non_standard_annotation",
    id="sentiment-1",
    value={
        "provider": "custom_ai",
        "label": "positive",
        "confidence": 0.96,
    },
)

print("Annotation type:", annotation["type"])
print("Sentiment:", annotation["value"]["label"])
print("Confidence:", annotation["value"]["confidence"])

# Here, value stores custom annotation data that does not fit a standard LangChain annotation type

Annotation type: non_standard_annotation
Sentiment: positive
Confidence: 0.96


# TextContentBlock: `TypedDict`
`TextContentBlock` represents normal text input or output in a message.
## Fields
1. `type`:`Literal["text"]`:= Identifies the block as text.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier.
3. `text`:`str` Stores the text content.
4. `annotations`:`NotRequired[list[Annotation]]`:= Stores citations and other annotations associated with the text.
5. `index`:`NotRequired[int | str]` Stores the block position in an aggregated or streamed response.
6. `extras`:`NotRequired[dict[str, Any]]` Stores provider-specific metadata.

In [6]:
from langchain_core.messages import Citation, TextContentBlock # Imports the required message types.

citation = Citation( # Creates a citation annotation.
    type="citation", # Identifies it as a citation.
    title="Python Documentation", # Stores the source title.
    url="https://docs.python.org/3/", # Stores the source URL.
    start_index=0, # Citation starts at the first character.
    end_index=27, # Citation ends at character index 27.
    cited_text="Python is a programming language", # Stores the cited source text.
)

text_block = TextContentBlock( # Creates a text content block.
    type="text", # Identifies the block as text.
    text="Python is a programming language.", # Stores the main text.
    annotations=[citation], # Attaches the citation to the text.
)

print(text_block["text"]) # Prints the text content.
print(text_block["annotations"][0]["url"]) # Prints the citation URL.

Python is a programming language.
https://docs.python.org/3/


# ToolCall:`TypedDict`
`ToolCall` represents a model request to invoke a tool using a name and structured arguments.
## Fields
1. `type`:`Literal["tool_call"]` Identifies the block as a tool call.
2. `id`:`str | None`:= Stores the tool-call identifier used to associate the call with its result.
3. `name`:`str`:= Stores the name of the tool to invoke.
4. `args`:`dict[str, Any]`:= Stores the arguments passed to the tool.
5. `index`:`NotRequired[int | str]`:= Stores the tool-call position in an aggregated or streamed response.
6. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.

In [7]:
from langchain_core.messages import ToolCall # Imports the ToolCall TypedDict.

tool_call = ToolCall( # Creates a request to call the add tool.
    type="tool_call", # Identifies this block as a tool call.
    id="call_1", # Connects the call with its future tool result.
    name="add", # Specifies the tool to execute.
    args={"a": 10, "b": 20}, # Supplies structured arguments to the tool.
    index=0, # Marks its position among multiple tool calls.
    extras={"provider": "custom_model"}, # Stores provider-specific metadata.
)

def add(a: int, b: int) -> int: # Defines the tool implementation.
    return a + b # Returns the sum.

result = add(**tool_call["args"]) # Executes the tool using the stored arguments.
print(result) # Output: 30

30


# ToolCallChunk:`TypedDict`
`ToolCallChunk` represents a partial tool call produced during streaming. Compatible chunks can be merged when they have the same non-null index.
## Fields
1. `type`:`Literal["tool_call_chunk"]`:= Identifies the block as a tool-call chunk.
2. `id`:`str | None`:= Stores the tool-call identifier.
3. `name`:`str | None`:= Stores a partial or complete tool name.
4. `args`:`str | None`:= Stores a partial JSON argument string.
5. `index`:`NotRequired[int | str]`:= Stores the position of the tool call in a sequence.
6. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.


In [8]:
from langchain_core.messages import ToolCallChunk # Imports the ToolCallChunk TypedDict.

chunk1 = ToolCallChunk( # Creates the first streamed tool-call part.
    type="tool_call_chunk", # Identifies it as a tool-call chunk.
    id="call_1", # Stores the tool-call identifier.
    name="get_", # Stores the first part of the tool name.
    args='{"city": "', # Stores the first part of the JSON arguments.
    index=0, # Marks this as tool call number 0.
)

chunk2 = ToolCallChunk( # Creates the second streamed tool-call part.
    type="tool_call_chunk", # Identifies it as a tool-call chunk.
    id=None, # The ID may be absent in later chunks.
    name="weather", # Stores the remaining part of the tool name.
    args='Delhi"}', # Stores the remaining JSON argument text.
    index=0, # Same index means both chunks belong to the same tool call.
)

tool_name = (chunk1["name"] or "") + (chunk2["name"] or "") # Merges the partial tool name.
tool_args = (chunk1["args"] or "") + (chunk2["args"] or "") # Merges the partial JSON arguments.

print(tool_name) # Output: get_weather
print(tool_args) # Output: {"city": "Delhi"}

get_weather
{"city": "Delhi"}


# InvalidToolCall:`TypedDict`
`InvalidToolCall` represents a tool call that could not be parsed or validated successfully.
## Fields
1. `type`:`Literal["invalid_tool_call"]`:= Identifies the block as an invalid tool call.
2. `id`:`str | None`:= Stores the tool-call identifier.
3. `name`:`str | None`:= Stores the tool name when available.
4. `args`:`str | None`:= Stores the unparsed tool arguments.
5. `error`:`str | None`:= Stores the parsing or generation error.
6. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
7. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.

In [9]:
from langchain_core.messages import InvalidToolCall # Imports the InvalidToolCall TypedDict.

invalid_call = InvalidToolCall( # Stores a tool call that could not be parsed.
    type="invalid_tool_call", # Identifies the block as an invalid tool call.
    id="call_2", # Stores the original tool-call identifier.
    name="get_weather", # Stores the requested tool name.
    args='{"city": "Delhi"', # Stores malformed JSON arguments.
    error="Missing closing brace in JSON.", # Stores the parsing error.
    index=0, # Marks its position among multiple tool calls.
    extras={"provider": "custom_model"}, # Stores provider-specific metadata.
)

print("Tool:", invalid_call["name"]) # Prints the attempted tool name.
print("Arguments:", invalid_call["args"]) # Prints the unparsed arguments.
print("Error:", invalid_call["error"]) # Prints why the tool call is invalid.

Tool: get_weather
Arguments: {"city": "Delhi"
Error: Missing closing brace in JSON.


# ServerToolCall:`TypedDict`
`ServerToolCall` represents a tool call, such as code execution or web search, that is executed by the model provider's server.
## Fields
1. `type`:`Literal["server_tool_call"]`:= Identifies the block as a server-side tool call.
2. `id`:`str`:= Stores the server-tool-call identifier.
3. `name`:`str`:= Stores the name of the server-side tool.
4. `args`:`dict[str, Any]`:= Stores the tool arguments.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.


In [10]:
from langchain_core.messages import ServerToolCall # Imports the ServerToolCall TypedDict.

server_call = ServerToolCall( # Represents a web search executed by the model provider.
    type="server_tool_call", # Identifies the block as a server-side tool call.
    id="server_call_1", # Stores the unique server-tool-call identifier.
    name="web_search", # Specifies the provider-hosted tool to execute.
    args={"query": "Latest Python release"}, # Stores arguments sent to the server-side tool.
    index=0, # Marks its position in the model response.
    extras={"provider": "example_provider"}, # Stores provider-specific metadata.
)

print("Tool:", server_call["name"]) # Prints the server-side tool name.
print("Query:", server_call["args"]["query"]) # Prints the search query.
print("Call ID:", server_call["id"]) # Prints the tool-call identifier.

Tool: web_search
Query: Latest Python release
Call ID: server_call_1


# ServerToolCallChunk:`TypedDict`
`ServerToolCallChunk` represents a partial server-side tool call produced during streaming.
## Fields
1. `type`:`Literal["server_tool_call_chunk"]`:= Identifies the block as a server-tool-call chunk.
2. `name`:`NotRequired[str]`:= Stores the server-side tool name when available.
3. `args`:`NotRequired[str]`:= Stores a JSON substring of the tool arguments.
4. `id`:`NotRequired[str]`:= Stores an optional unique identifier.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.

In [13]:
from langchain_core.messages import ServerToolCallChunk # Imports the ServerToolCallChunk TypedDict.

chunk1 = ServerToolCallChunk( # Creates the first streamed server-tool-call chunk.
    type="server_tool_call_chunk", # Identifies the block as a server-tool-call chunk.
    name="web_search", # Stores the server-side tool name.
    args='{"query": "LangChain ', # Stores the first JSON argument fragment.
    id="server_call_1", # Stores the server-tool-call identifier.
    index=0, # Marks which server tool call this chunk belongs to.
)

chunk2 = ServerToolCallChunk( # Creates the next chunk of the same server tool call.
    type="server_tool_call_chunk", # Identifies the block as a server-tool-call chunk.
    args='documentation"}', # Stores the remaining JSON argument fragment.
    index=0, # Same index means both chunks belong to the same call.
)

complete_args = chunk1.get("args", "") + chunk2.get("args", "") # Combines the streamed JSON fragments.

print(chunk1["name"]) # Output: web_search
print(complete_args) # Output: {"query": "LangChain documentation"}

web_search
{"query": "LangChain documentation"}


# ServerToolResult:`TypedDict`
`ServerToolResult` represents the result returned by a server-side tool.
## Fields
1. `type`:`Literal["server_tool_result"]`:= Identifies the block as a server-tool result.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier for the result block.
3. `tool_call_id`:`str`:= Stores the identifier of the corresponding server-side tool call.
4. `status`:`Literal["success", "error"]`:= Stores whether the server-side tool execution succeeded or failed.
5. `output`:`NotRequired[Any]`:= Stores the result returned by the tool.
6. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
7. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.


In [14]:
from langchain_core.messages import ServerToolResult # Imports the ServerToolResult TypedDict.

result = ServerToolResult( # Represents the result of a provider-side web search.
    type="server_tool_result", # Identifies the block as a server-tool result.
    id="result_1", # Stores the unique result-block identifier.
    tool_call_id="server_call_1", # Links the result to the original server tool call.
    status="success", # Indicates that the server-side tool completed successfully.
    output={"title": "LangChain Docs", "url": "https://python.langchain.com"}, # Stores the tool output.
    index=1, # Marks the result block position in the response.
    extras={"provider": "example_provider"}, # Stores provider-specific metadata.
)

print("Status:", result["status"]) # Prints the execution status.
print("Title:", result["output"]["title"]) # Prints the returned page title.
print("Call ID:", result["tool_call_id"]) # Prints the related server-tool-call ID.

Status: success
Title: LangChain Docs
Call ID: server_call_1


# ReasoningContentBlock:`TypedDict`
`ReasoningContentBlock` represents reasoning text, a thought summary, or raw reasoning output produced by a model.
## Fields
1. `type`:`Literal["reasoning"]`:= Identifies the block as reasoning.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier.
3. `reasoning`:`NotRequired[str]`:= Stores reasoning text or a thought summary.
4. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
5. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata.

In [15]:
from langchain_core.messages import ReasoningContentBlock # Imports the ReasoningContentBlock TypedDict.

reasoning_block = ReasoningContentBlock( # Creates a reasoning summary returned by the model.
    type="reasoning", # Identifies the block as reasoning content.
    id="reasoning_1", # Stores an optional unique block identifier.
    reasoning="The user asked for the larger value, so compare both numbers.", # Stores the reasoning summary.
    index=0, # Marks the block position in the response.
    extras={"provider": "example_provider"}, # Stores provider-specific metadata.
)

print("Reasoning:", reasoning_block["reasoning"]) # Prints the reasoning summary.
print("Block ID:", reasoning_block["id"]) # Prints the block identifier.

Reasoning: The user asked for the larger value, so compare both numbers.
Block ID: reasoning_1


# ImageContentBlock:`TypedDict`
`ImageContentBlock` represents image data supplied through a URL, base64 data, or an external file identifier.
## Fields
1. `type`:`Literal["image"]`:= Identifies the block as an image.
2. `id`:`NotRequired[str]`:= Stores an optional unique block identifier.
3. `file_id`:`NotRequired[str]`:= Stores a reference to an image in an external file-storage system.
4. `mime_type`:`NotRequired[str]`:= Stores the image MIME type.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `url`:`NotRequired[str]`:= Stores the image URL.
7. `base64`:`NotRequired[str]`:= Stores base64-encoded image data.
8. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata that is not the image data itself.


In [16]:
from langchain_core.messages import ImageContentBlock # Imports the ImageContentBlock TypedDict.

image_block = ImageContentBlock( # Creates an image content block using a public URL.
    type="image", # Identifies the block as image content.
    id="image_1", # Stores an optional unique block identifier.
    url="https://example.com/photo.png", # Stores the image URL.
    mime_type="image/png", # Specifies the image format.
    index=0, # Marks the image position in the message content.
    extras={"alt_text": "A mountain landscape"}, # Stores provider-specific metadata.
)

print("Image URL:", image_block["url"]) # Prints the image location.
print("MIME type:", image_block["mime_type"]) # Prints the image format.
print("Description:", image_block["extras"]["alt_text"]) # Prints the custom metadata.

Image URL: https://example.com/photo.png
MIME type: image/png
Description: A mountain landscape


# VideoContentBlock:`TypedDict`
`VideoContentBlock` represents video data supplied through a URL, base64 data, or an external file identifier.
## Fields
1. `type`:`Literal["video"]`:= Identifies the block as video.
2. `id`:`NotRequired[str]`:= Stores an optional unique block identifier.
3. `file_id`:`NotRequired[str]`:= Stores a reference to a video in an external file-storage system.
4. `mime_type`:`NotRequired[str]`:= Stores the video MIME type.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `url`:`NotRequired[str]`:= Stores the video URL.
7. `base64`:`NotRequired[str]`:= Stores base64-encoded video data.
8. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata that is not the video data itself.

In [17]:
from langchain_core.messages import VideoContentBlock # Imports the VideoContentBlock TypedDict.

video_block = VideoContentBlock( # Creates a video content block using a public URL.
    type="video", # Identifies the block as video content.
    id="video_1", # Stores an optional unique block identifier.
    url="https://example.com/tutorial.mp4", # Stores the video URL.
    mime_type="video/mp4", # Specifies the video format.
    index=0, # Marks the video position in the message content.
    extras={"duration_seconds": 120}, # Stores provider-specific metadata.
)

print("Video URL:", video_block["url"]) # Prints the video location.
print("MIME type:", video_block["mime_type"]) # Prints the video format.
print("Duration:", video_block["extras"]["duration_seconds"]) # Prints the custom duration metadata.

Video URL: https://example.com/tutorial.mp4
MIME type: video/mp4
Duration: 120


# AudioContentBlock:`TypedDict`
`AudioContentBlock` represents audio data supplied through a URL, base64 data, or an external file identifier.
## Fields
1. `type`:`Literal["audio"]`:= Identifies the block as audio.
2. `id`:`NotRequired[str]`:= Stores an optional unique block identifier.
3. `file_id`:`NotRequired[str]`:= Stores a reference to audio in an external file-storage system.
4. `mime_type`:`NotRequired[str]`:= Stores the audio MIME type.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `url`:`NotRequired[str]`:= Stores the audio URL.
7. `base64`:`NotRequired[str]`:= Stores base64-encoded audio data.
8. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata that is not the audio data itself.

In [18]:
from langchain_core.messages import AudioContentBlock # Imports the AudioContentBlock TypedDict.

audio_block = AudioContentBlock( # Creates an audio content block using a public URL.
    type="audio", # Identifies the block as audio content.
    id="audio_1", # Stores an optional unique block identifier.
    url="https://example.com/podcast.mp3", # Stores the audio URL.
    mime_type="audio/mpeg", # Specifies the audio format.
    index=0, # Marks the audio position in the message content.
    extras={"duration_seconds": 180}, # Stores provider-specific metadata.
)

print("Audio URL:", audio_block["url"]) # Prints the audio location.
print("MIME type:", audio_block["mime_type"]) # Prints the audio format.
print("Duration:", audio_block["extras"]["duration_seconds"]) # Prints the custom duration metadata.

Audio URL: https://example.com/podcast.mp3
MIME type: audio/mpeg
Duration: 180


# PlainTextContentBlock:`TypedDict`
`PlainTextContentBlock` represents plaintext data, such as content from a `.txt` or `.md` document.
## Fields
1. `type`:`Literal["text-plain"]`:= Identifies the block as plaintext.
2. `id`:`NotRequired[str]`:= Stores an optional unique block identifier.
3. `file_id`:`NotRequired[str]`:= Stores a reference to plaintext in an external file-storage system.
4. `mime_type`:`Literal["text/plain"]`:= Stores the fixed plaintext MIME type.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `url`:`NotRequired[str]`:= Stores the plaintext document URL.
7. `base64`:`NotRequired[str]`:= Stores base64-encoded plaintext data.
8. `text`:`NotRequired[str]`:= Stores plaintext directly.
9. `title`:`NotRequired[str]`:= Stores an optional title for the text data.
10. `context`:`NotRequired[str]`:= Stores an optional description or summary of the text content.
11. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata that is not the plaintext data itself.

In [19]:
from langchain_core.messages import PlainTextContentBlock # Imports the PlainTextContentBlock TypedDict.

text_block = PlainTextContentBlock( # Creates a plaintext document block.
    type="text-plain", # Identifies the block as plaintext.
    id="text_file_1", # Stores an optional unique block identifier.
    mime_type="text/plain", # Uses the fixed plaintext MIME type.
    text="LangChain helps build applications using language models.", # Stores the document text directly.
    title="LangChain Notes", # Stores an optional document title.
    context="Short notes about LangChain.", # Stores a brief description of the text.
    index=0, # Marks the block position in the message content.
    extras={"source": "notes.txt"}, # Stores provider-specific metadata.
)

print("Title:", text_block["title"]) # Prints the document title.
print("Text:", text_block["text"]) # Prints the plaintext content.
print("Source:", text_block["extras"]["source"]) # Prints the custom source metadata.

Title: LangChain Notes
Text: LangChain helps build applications using language models.
Source: notes.txt


# FileContentBlock:`TypedDict`
`FileContentBlock` represents file data that does not fit the image, audio, video, or plaintext block types, such as PDF or Word documents.
## Fields
1. `type`:`Literal["file"]`:= Identifies the block as a file.
2. `id`:`NotRequired[str]`:= Stores an optional unique identifier for the content block.
3. `file_id`:`NotRequired[str]`:= Stores a reference to the file in an external file-storage system.
4. `mime_type`:`NotRequired[str]`:= Stores the MIME type of the file.
5. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.
6. `url`:`NotRequired[str]`:= Stores the file URL.
7. `base64`:`NotRequired[str]`:= Stores base64-encoded file data.
8. `extras`:`NotRequired[dict[str, Any]]`:= Stores provider-specific metadata that is not the file data itself.

In [20]:
from langchain_core.messages import FileContentBlock # Imports the FileContentBlock TypedDict.

file_block = FileContentBlock( # Creates a file block for a PDF document.
    type="file", # Identifies the block as generic file content.
    id="file_1", # Stores an optional unique block identifier.
    url="https://example.com/report.pdf", # Stores the file URL.
    mime_type="application/pdf", # Specifies that the file is a PDF.
    index=0, # Marks the file position in the message content.
    extras={"filename": "report.pdf", "pages": 12}, # Stores file-related custom metadata.
)

print("File URL:", file_block["url"]) # Prints the file location.
print("File type:", file_block["mime_type"]) # Prints the MIME type.
print("Filename:", file_block["extras"]["filename"]) # Prints the filename.

File URL: https://example.com/report.pdf
File type: application/pdf
Filename: report.pdf


# NonStandardContentBlock:`TypedDict`
`NonStandardContentBlock` stores provider-specific content that does not yet have a standardized block type.
## Fields
1. `type`:`Literal["non_standard"]`:= Identifies the block as non-standard.
2. `id`:`NotRequired[str]`:= Stores an optional unique block identifier.
3. `value`:`dict[str, Any]`:= Stores provider-specific content data.
4. `index`:`NotRequired[int | str]`:= Stores the block position in an aggregated or streamed response.

In [21]:
from langchain_core.messages import NonStandardContentBlock # Imports the NonStandardContentBlock TypedDict.

custom_block = NonStandardContentBlock( # Creates a provider-specific content block.
    type="non_standard", # Identifies the block as non-standard content.
    id="custom_1", # Stores an optional unique block identifier.
    value={ # Stores provider-specific data.
        "provider": "custom_model",
        "chart_type": "bar",
        "data": [10, 20, 30],
    },
    index=0, # Marks the block position in the message content.
)

print("Provider:", custom_block["value"]["provider"]) # Prints the provider name.
print("Chart type:", custom_block["value"]["chart_type"]) # Prints the custom content type.
print("Data:", custom_block["value"]["data"]) # Prints the provider-specific data.

Provider: custom_model
Chart type: bar
Data: [10, 20, 30]


## Functions

1. `is_data_content_block`: Checks whether a dictionary is an old-style or new-style multimodal data content block.
   * **Syntax:**
     ```python
     is_data_content_block(
         block: dict[str, Any] # Content block to inspect
     ) -> bool
     ```

In [22]:
from langchain_core.messages import is_data_content_block # Imports the block-checking function.

new_image = {"type": "image", "url": "https://example.com/photo.png"} # New-style image data block.
old_image = {"type": "image", "source_type": "base64", "data": "abc123"} # Old-style image data block.
text_block = {"type": "text", "text": "Hello"} # Normal text block, not multimodal data.

print(is_data_content_block(new_image)) # Output: True
print(is_data_content_block(old_image)) # Output: True
print(is_data_content_block(text_block)) # Output: False

True
True
False


2. `create_text_block`: Creates a standardized text content block.

   An identifier is generated automatically when one is not provided. Additional non-null keyword arguments are stored in `extras`.

   * **Syntax:**
     ```python
     create_text_block(
         text: str, # Text content
         *,
         id: str | None = None, # Optional block identifier
         annotations: list[Annotation] | None = None, # Citations and other annotations
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> TextContentBlock
     ```

In [35]:
from langchain_core.messages.content import Citation, create_text_block # Imports the helper function and citation type.

citation = Citation( # Creates a citation attached to the text.
    type="citation", # Identifies the annotation as a citation.
    title="Python Documentation", # Stores the source title.
    url="https://docs.python.org/3/", # Stores the source URL.
    start_index=0, # Marks where the citation starts in the response text.
    end_index=27, # Marks where the citation ends in the response text.
)

text_block = create_text_block( # Creates a standardized text content block.
    "Python is a programming language.", # Stores the main text.
    annotations=[citation], # Attaches the citation to the text.
    index=0, # Stores the block position.
    provider="custom_model", # Stored automatically inside extras.
)

print(text_block["text"]) # Output: Python is a programming language.
print(text_block["id"]) # Prints the automatically generated block ID.
print(text_block["extras"]["provider"]) # Output: custom_model

Python is a programming language.
lc_8dc458a4-8cb8-4799-8b9c-af4e3641f2fc
custom_model


3. `create_image_block`: Creates an image content block from a URL, base64 data, or external file identifier.

   At least one image source must be supplied. An identifier is generated automatically when omitted.

   * **Syntax:**
     ```python
     create_image_block(
         *,
         url: str | None = None, # Image URL
         base64: str | None = None, # Base64-encoded image data
         file_id: str | None = None, # External image-file identifier
         mime_type: str | None = None, # Image MIME type
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> ImageContentBlock
     ```

In [36]:
from langchain_core.messages.content import create_image_block # Imports the image-block helper function.

image_block = create_image_block( # Creates a standardized image content block.
    url="https://example.com/photo.png", # Uses an image URL as the source.
    mime_type="image/png", # Specifies the image format.
    index=0, # Stores the block position in the message.
    alt_text="Mountain landscape", # Stored automatically inside extras.
)

print(image_block["url"]) # Output: https://example.com/photo.png
print(image_block["id"]) # Prints the automatically generated block ID.
print(image_block["extras"]["alt_text"]) # Output: Mountain landscape

https://example.com/photo.png
lc_20db1b11-3d6b-43de-8f6c-0d0973ed7738
Mountain landscape


4. `create_video_block`: Creates a video content block from a URL, base64 data, or external file identifier.

   At least one video source must be supplied. `mime_type` is required when base64 data is used.

   * **Syntax:**
     ```python
     create_video_block(
         *,
         url: str | None = None, # Video URL
         base64: str | None = None, # Base64-encoded video data
         file_id: str | None = None, # External video-file identifier
         mime_type: str | None = None, # Video MIME type
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> VideoContentBlock
     ```

In [37]:
from langchain_core.messages.content import create_video_block # Imports the video-block helper function.

video_block = create_video_block( # Creates a standardized video content block.
    url="https://example.com/tutorial.mp4", # Uses a video URL as the source.
    mime_type="video/mp4", # Specifies the video format.
    index=0, # Stores the block position in the message.
    duration_seconds=120, # Stored automatically inside extras.
)

print(video_block["url"]) # Output: https://example.com/tutorial.mp4
print(video_block["id"]) # Prints the automatically generated block ID.
print(video_block["extras"]["duration_seconds"]) # Output: 120

https://example.com/tutorial.mp4
lc_d28db0e5-5894-4b56-b293-c6e80e7f9254
120


5. `create_audio_block`: Creates an audio content block from a URL, base64 data, or external file identifier.

   At least one audio source must be supplied. `mime_type` is required when base64 data is used.

   * **Syntax:**
     ```python
     create_audio_block(
         *,
         url: str | None = None, # Audio URL
         base64: str | None = None, # Base64-encoded audio data
         file_id: str | None = None, # External audio-file identifier
         mime_type: str | None = None, # Audio MIME type
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> AudioContentBlock
     ```

In [38]:
from langchain_core.messages.content import create_audio_block # Imports the audio-block helper function.

audio_block = create_audio_block( # Creates a standardized audio content block.
    url="https://example.com/podcast.mp3", # Uses an audio URL as the source.
    mime_type="audio/mpeg", # Specifies the audio format.
    index=0, # Stores the block position in the message.
    duration_seconds=180, # Stored automatically inside extras.
)

print(audio_block["url"]) # Output: https://example.com/podcast.mp3
print(audio_block["id"]) # Prints the automatically generated block ID.
print(audio_block["extras"]["duration_seconds"]) # Output: 180

https://example.com/podcast.mp3
lc_6fb17e57-9408-48ca-a8b6-2b1661e0d07d
180


6. `create_file_block`: Creates a file content block from a URL, base64 data, or external file identifier.

   At least one file source must be supplied. `mime_type` is required when base64 data is used.

   * **Syntax:**
     ```python
     create_file_block(
         *,
         url: str | None = None, # File URL
         base64: str | None = None, # Base64-encoded file data
         file_id: str | None = None, # External file identifier
         mime_type: str | None = None, # File MIME type
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> FileContentBlock
     ```

In [39]:
from langchain_core.messages.content import create_file_block # Imports the file-block helper function.

file_block = create_file_block( # Creates a standardized file content block.
    url="https://example.com/report.pdf", # Uses a file URL as the source.
    mime_type="application/pdf", # Specifies that the file is a PDF.
    index=0, # Stores the block position in the message.
    filename="report.pdf", # Stored automatically inside extras.
)

print(file_block["url"]) # Output: https://example.com/report.pdf
print(file_block["id"]) # Prints the automatically generated block ID.
print(file_block["extras"]["filename"]) # Output: report.pdf

https://example.com/report.pdf
lc_90fa17f6-4beb-4c5e-924d-ce7aabb9e998
report.pdf


7. `create_plaintext_block`: Creates a plaintext content block.

   The plaintext may be supplied directly, by URL, as base64 data, or by external file identifier. The MIME type is set to `text/plain`.

   * **Syntax:**
     ```python
     create_plaintext_block(
         text: str | None = None, # Plaintext content
         url: str | None = None, # Plaintext URL
         base64: str | None = None, # Base64-encoded plaintext data
         file_id: str | None = None, # External plaintext-file identifier
         title: str | None = None, # Optional document title
         context: str | None = None, # Optional description or summary
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> PlainTextContentBlock
     ```

In [40]:
from langchain_core.messages.content import create_plaintext_block # Imports the plaintext-block helper function.

text_block = create_plaintext_block( # Creates a standardized plaintext content block.
    text="LangChain helps build applications using language models.", # Stores the plaintext directly.
    title="LangChain Notes", # Stores an optional document title.
    context="A short introduction to LangChain.", # Stores a description of the document.
    index=0, # Stores the block position in the message.
    filename="notes.txt", # Stored automatically inside extras.
)

print(text_block["text"]) # Prints the plaintext content.
print(text_block["mime_type"]) # Output: text/plain
print(text_block["id"]) # Prints the automatically generated block ID.
print(text_block["extras"]["filename"]) # Output: notes.txt

LangChain helps build applications using language models.
text/plain
lc_b99df296-e289-4178-b180-618bab8e2cb8
notes.txt


8. `create_tool_call`: Creates a standardized tool-call block.

   An identifier is generated automatically when one is not provided. Additional non-null keyword arguments are stored in `extras`.

   * **Syntax:**
     ```python
     create_tool_call(
         name: str, # Name of the tool to invoke
         args: dict[str, Any], # Tool arguments
         *,
         id: str | None = None, # Optional tool-call identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> ToolCall
     ```

In [41]:
from langchain_core.messages.content import create_tool_call # Imports the tool-call helper function.

tool_call = create_tool_call( # Creates a standardized request to call a tool.
    name="add", # Specifies the tool to invoke.
    args={"a": 10, "b": 20}, # Stores the arguments passed to the tool.
    index=0, # Stores the tool-call position in the response.
    provider="custom_model", # Stored automatically inside extras.
)

def add(a: int, b: int) -> int: # Defines the tool implementation.
    return a + b # Returns the sum.

result = add(**tool_call["args"]) # Executes the tool using the stored arguments.

print(tool_call["id"]) # Prints the automatically generated tool-call ID.
print(tool_call["extras"]["provider"]) # Output: custom_model
print(result) # Output: 30

lc_96ff462f-6f0d-4e13-96aa-92d17e2df452
custom_model
30


9. `create_reasoning_block`: Creates a standardized reasoning content block.

   An identifier is generated automatically when one is not provided. Missing reasoning text is stored as an empty string.

   * **Syntax:**
     ```python
     create_reasoning_block(
         reasoning: str | None = None, # Reasoning text or thought summary
         id: str | None = None, # Optional block identifier
         index: int | str | None = None, # Position in an aggregated response
         **kwargs: Any # Provider-specific metadata
     ) -> ReasoningContentBlock
     ```

In [42]:
from langchain_core.messages.content import create_reasoning_block # Imports the reasoning-block helper function.

reasoning_block = create_reasoning_block( # Creates a standardized reasoning content block.
    reasoning="The user asked for the larger number, so compare both values.", # Stores the reasoning summary.
    index=0, # Stores the block position in the response.
    provider="custom_model", # Stored automatically inside extras.
)

print(reasoning_block["reasoning"]) # Prints the reasoning text.
print(reasoning_block["id"]) # Prints the automatically generated block ID.
print(reasoning_block["extras"]["provider"]) # Output: custom_model

The user asked for the larger number, so compare both values.
lc_de6806be-296d-4713-a2a3-c1f1902e9755
custom_model


10. `create_citation`: Creates a standardized citation annotation.

    An identifier is generated automatically when one is not provided. Additional non-null keyword arguments are stored in `extras`.

    * **Syntax:**
      ```python
      create_citation(
          *,
          url: str | None = None, # Source URL
          title: str | None = None, # Source title
          start_index: int | None = None, # Start index in the response text
          end_index: int | None = None, # End index in the response text
          cited_text: str | None = None, # Excerpt from the source
          id: str | None = None, # Optional citation identifier
          **kwargs: Any # Provider-specific metadata
      ) -> Citation
      ```

In [43]:
from langchain_core.messages.content import create_citation # Imports the citation helper function.

response = "Paris is the capital of France." # Stores the model response text.
start = response.index("Paris") # Finds where the cited text begins.
end = start + len("Paris") # Calculates where the cited text ends.

citation = create_citation( # Creates a standardized citation annotation.
    url="https://en.wikipedia.org/wiki/Paris", # Stores the source URL.
    title="Paris", # Stores the source title.
    start_index=start, # Marks the citation start in the response text.
    end_index=end, # Marks the citation end in the response text.
    cited_text="Paris", # Stores the cited source excerpt.
    provider="custom_model", # Stored automatically inside extras.
)

print(response[citation["start_index"]:citation["end_index"]]) # Output: Paris
print(citation["url"]) # Prints the source URL.
print(citation["id"]) # Prints the automatically generated citation ID.

Paris
https://en.wikipedia.org/wiki/Paris
lc_c97c80f4-a238-4635-9cfc-6c6bb2e1c95d


11. `create_non_standard_block`: Creates a provider-specific non-standard content block.

    An identifier is generated automatically when one is not provided.

    * **Syntax:**
      ```python
      create_non_standard_block(
          value: dict[str, Any], # Provider-specific content data
          *,
          id: str | None = None, # Optional block identifier
          index: int | str | None = None # Position in an aggregated response
      ) -> NonStandardContentBlock
      ```

In [34]:
from langchain_core.messages.content import create_non_standard_block # Imports the helper function.

custom_block = create_non_standard_block( # Creates a provider-specific content block.
    value={ # Stores content that has no standard LangChain block type.
        "provider": "custom_model",
        "chart_type": "bar",
        "data": [10, 20, 30],
    },
    index=0, # Stores the block position in the response.
)

print(custom_block["type"]) # Output: non_standard
print(custom_block["id"]) # Prints the automatically generated block ID.
print(custom_block["value"]["chart_type"]) # Output: bar
print(custom_block["value"]["data"]) # Output: [10, 20, 30]

non_standard
lc_01b82f52-40ac-4fc1-8188-82ddf3c23b04
bar
[10, 20, 30]


## Type Aliases

1. `Annotation`: Represents all supported annotation types.
   * **Definition:**
     ```python
     Annotation = Citation | NonStandardAnnotation
     ```

2. `DataContentBlock`: Represents all standardized multimodal data-block types.
   * **Definition:**
     ```python
     DataContentBlock = (
         ImageContentBlock
         | VideoContentBlock
         | AudioContentBlock
         | PlainTextContentBlock
         | FileContentBlock
     )
     ```

3. `ToolContentBlock`: Represents all supported tool-related content blocks.
   * **Definition:**
     ```python
     ToolContentBlock = (
         ToolCall
         | ToolCallChunk
         | ServerToolCall
         | ServerToolCallChunk
         | ServerToolResult
     )
     ```

4. `ContentBlock`: Represents all standardized message content-block types.
   * **Definition:**
     ```python
     ContentBlock = (
         TextContentBlock
         | InvalidToolCall
         | ReasoningContentBlock
         | NonStandardContentBlock
         | DataContentBlock
         | ToolContentBlock
     )
     ```

In [44]:
from langchain_core.messages.content import (
    Annotation, Citation,
    DataContentBlock, ImageContentBlock,
    ToolContentBlock, ToolCall,
    ContentBlock, TextContentBlock,
)

annotation: Annotation = Citation( # Accepts Citation or NonStandardAnnotation.
    type="citation",
    title="Python Docs",
    url="https://docs.python.org/3/",
)

data_block: DataContentBlock = ImageContentBlock( # Accepts image, video, audio, plaintext, or file blocks.
    type="image",
    url="https://example.com/photo.png",
)

tool_block: ToolContentBlock = ToolCall( # Accepts supported tool-related blocks.
    type="tool_call",
    id="call_1",
    name="add",
    args={"a": 10, "b": 20},
)

content_block: ContentBlock = TextContentBlock( # Accepts any standardized content-block type.
    type="text",
    text="Hello from LangChain",
    annotations=[annotation],
)

print(content_block["text"]) # Output: Hello from LangChain
print(tool_block["name"]) # Output: add
print(data_block["url"]) # Output: https://example.com/photo.png

Hello from LangChain
add
https://example.com/photo.png
